# Topic Modeling: BERTopic + WangchanBERTa + topics_over_time

**เป้าหมาย:** หา "เรื่องเดียวกัน" ข้ามรัฐธรรมนูญทุกฉบับ โดยไม่พึ่ง chapter/section number (ซึ่งไม่ตรงกัน)

**Pipeline**
1. โหลด `sections_v2.csv` → 1 row = 1 มาตรา
2. แยกข้อความ 2 ชุด:
   - `docs_for_embeddings` = raw text (ส่งให้ WangchanBERTa จับ semantic)
   - `docs_for_ctfidf` = tokens หลังลบ Thai stopwords + **legal noise** (ใช้สกัด keyword ของ topic)
3. Embed ด้วย **WangchanBERTa** (`airesearch/wangchanberta-base-att-spm-uncased`) + mean-pooling + L2 normalize
4. UMAP → HDBSCAN → c-TF-IDF (BERTopic)
5. `topics_over_time()` ดูว่าแต่ละ topic ปรากฏบ่อยแค่ไหนในแต่ละปี พ.ศ.

**ความแตกต่างหลักจาก `topic_modeling_mpnet.ipynb`**
- เปลี่ยน embedding จาก `all-mpnet-base-v2` (English-first) → WangchanBERTa (Thai-native)
- ขยาย legal stopwords + แก้บั๊กที่ `legal_noise` ไม่ถูก merge
- เพิ่ม `topics_over_time()` วิเคราะห์เชิงเวลา
- กำหนด `random_state` + `min_cluster_size` ให้ผลรันซ้ำได้

In [152]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

font_path = '../../static/font/LINESeedSansTH_Rg.ttf'
fm.fontManager.addfont(font_path)
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
print(f"Font loaded: {font_prop.get_name()}")

Font loaded: LINE Seed Sans TH


In [153]:
%pip install pandas numpy scikit-learn pythainlp attacut bertopic transformers torch sentencepiece tqdm umap-learn hdbscan


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [154]:
import re
import warnings
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from pythainlp.corpus.common import thai_stopwords
from pythainlp.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoTokenizer, AutoModel

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: mps


## 1. โหลดข้อมูล + Tokenize

อ่านจาก `sections_v2.csv` แล้วเตรียม 2 ชุดข้อความตามที่อธิบายไว้ข้างบน

In [155]:
df = pd.read_csv('../output/sections_v2.csv', encoding='utf-8-sig', engine='python', on_bad_lines='skip')
df = df.rename(columns={'doc_id': 'constitution_id'})
df['text'] = df['text'].fillna('').astype(str)
df = df[df['text'].str.strip().ne('')].copy().reset_index(drop=True)

print(f"Total sections: {len(df)}")
print(f"Constitutions: {df['constitution_id'].nunique()}")
print(f"Year range: {df['year_th'].min()} - {df['year_th'].max()}")
df.head(3)

Total sections: 2797
Constitutions: 38
Year range: 2475 - 2564


,section_id,constitution_id,doc_type,year_th,year_ce,name_short,era,regime_type,parent_doc_id,chapter_number,chapter_title,sub_section_number,sub_section_title,section_number,section_role,target_chapter,target_section_no,change_mode,text
0,const_2475_s_1,const_2475,full,2475,1932,Constitution 2475,early_democracy,civilian,NaN,0,บททั่วไป,NaN,NaN,1,content,NaN,NaN,content,สยามประเทศเป็นราชอาณาจักรอันหนึ่งอันเดียว จะแบ...
1,const_2475_s_2,const_2475,full,2475,1932,Constitution 2475,early_democracy,civilian,NaN,0,บททั่วไป,NaN,NaN,2,content,NaN,NaN,content,อำนาจอธิปไตยย่อมมาจากปวงชนชาวสยาม พระมหากษัตริ...
2,const_2475_s_3,const_2475,full,2475,1932,Constitution 2475,early_democracy,civilian,NaN,1,พระมหากษัตริย์,NaN,NaN,3,content,NaN,NaN,content,องค์พระมหากษัตริย์ดำรงอยู่ในฐานะอันเป็นที่เคาร...


In [156]:
# Legal stopwords แบ่งเป็น 3 กลุ่มตาม design
# A: คำเชื่อม/ฟังก์ชัน — ลบแน่นอน
GROUP_A_FUNCTION = [
    'ตาม', 'แห่ง', 'ว่าด้วย', 'ทั้งนี้', 'ดังกล่าว', 'ดังต่อไปนี้', 'อันเป็น',
    'โดย', 'ซึ่ง', 'อัน', 'ใน', 'แก่', 'ต่อ', 'จาก', 'ระหว่าง', 'เพื่อ',
    'มิ', 'ย่อม', 'ให้', 'แต่', 'หรือ', 'และ', 'รวมทั้ง', 'อีก', 'ทั้ง',
]

# B: คำโครงสร้างเอกสาร — ลบแน่นอน (ไม่ใช่ topic)
GROUP_B_STRUCTURE = [
    'มาตรา', 'หมวด', 'วรรค', 'ส่วน', 'บท', 'บัญญัติ', 'ฉบับ', 'พ.ศ.',
    'พุทธศักราช', 'รัฐธรรมนูญ', 'ราชอาณาจักรไทย', 'ประกาศ',
]

# C: noise ระดับกลาง — เปิดแล้วเพราะ topic-info แสดง 'กฎหมาย'/'บัญญัติ'/'หน้าที่' โผล่ในหลาย topic
# หมายเหตุ: 'พระมหากษัตริย์', 'รัฐมนตรี', 'สภา', 'อำนาจ' ไม่อยู่ที่นี่ — เป็น topic จริง
GROUP_C_NOISE = [
    'กฎหมาย', 'ระเบียบ', 'ภายใต้', 'กรณี', 'หน้าที่', 'ใด', 'ท่าน',
    'บังคับ', 'พระ', 'บรรดา',
]

USE_GROUP_C = True  # เปิดเพื่อกำจัด noise words ที่กลายเป็น top word ของหลาย topic

legal_noise = set(GROUP_A_FUNCTION) | set(GROUP_B_STRUCTURE)
if USE_GROUP_C:
    legal_noise |= set(GROUP_C_NOISE)

custom_stopwords = sorted(set(thai_stopwords()) | legal_noise)
print(f"Total stopwords: {len(custom_stopwords)} (legal additions: {len(legal_noise)})")

Total stopwords: 1053 (legal additions: 47)


In [157]:
stopword_set = set(custom_stopwords)

def preprocess_thai(text: str) -> list[str]:
    tokens = word_tokenize(text, engine='attacut')
    return [
        t.strip() for t in tokens
        if t.strip() and t.strip() not in stopword_set and not re.match(r'^[\W_]+$', t.strip())
    ]

tqdm.pandas(desc='Tokenizing')
df['tokens'] = df['text'].progress_apply(preprocess_thai)
df['joined_tokens'] = df['tokens'].apply(lambda xs: ' '.join(xs))
df['word_count'] = df['tokens'].apply(len)

# เก็บมาตราสั้นไว้ด้วย เพราะหลายมาตราสั้นเป็นหลักการสำคัญของรัฐธรรมนูญ
# เช่น อำนาจอธิปไตย การแบ่งแยกราชอาณาจักร หรือหน้าที่พื้นฐาน
# ถ้าต้องการวิเคราะห์เฉพาะข้อความยาว ให้ filter จาก word_count ภายหลังแทน
MIN_TOKENS = 1
short_sections = int((df['word_count'] < 8).sum())
print(f"Kept {len(df)} sections; {short_sections} have < 8 cleaned tokens")
df[['section_id', 'year_th', 'text', 'joined_tokens']].head(3)

Tokenizing: 100%|██████████| 2797/2797 [00:13<00:00, 201.15it/s]

Kept 2797 sections; 478 have < 8 cleaned tokens


,section_id,year_th,text,joined_tokens
0,const_2475_s_1,2475,สยามประเทศเป็นราชอาณาจักรอันหนึ่งอันเดียว จะแบ...,สยาม ประเทศ ราชอาณาจักร แบ่งแยก ประชาชน สยาม ก...
1,const_2475_s_2,2475,อำนาจอธิปไตยย่อมมาจากปวงชนชาวสยาม พระมหากษัตริ...,อำนาจ อธิปไตย ปวง ชน สยาม พระมหากษัตริย์ ประมุ...
2,const_2475_s_3,2475,องค์พระมหากษัตริย์ดำรงอยู่ในฐานะอันเป็นที่เคาร...,องค์ พระมหากษัตริย์ ดำรง ฐานะ เคารพ สัก ละเมิด


## 2. Embedding ด้วย WangchanBERTa

WangchanBERTa เป็น MLM (RoBERTa-base) เทรนบนภาษาไทย 78GB — ไม่ใช่ sentence encoder โดยตรง
ดังนั้นต้องทำ **mean-pooling แบบมี attention mask** + **L2 normalize** เอง เพื่อให้ได้ sentence vector ที่ใช้ cosine similarity ได้

Max length = 416 (SPM) — มาตราที่ยาวกว่านี้จะถูก truncate

In [158]:
MODEL_NAME = 'airesearch/wangchanberta-base-att-spm-uncased'
MAX_LENGTH = 416
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print(f"Loaded {MODEL_NAME} on {DEVICE}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7781.42it/s]
[transformers] CamembertModel LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded airesearch/wangchanberta-base-att-spm-uncased on mps


In [159]:
@torch.no_grad()
def embed_batch(texts: list[str]) -> np.ndarray:
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt',
    ).to(DEVICE)
    out = model(**enc).last_hidden_state  # (B, T, H)

    # mean pool with attention mask
    mask = enc['attention_mask'].unsqueeze(-1).float()  # (B, T, 1)
    summed = (out * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    pooled = summed / counts  # (B, H)

    # L2 normalize so cosine == dot product
    pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
    return pooled.cpu().numpy().astype('float32')


def embed_corpus(texts: list[str], batch_size: int = BATCH_SIZE) -> np.ndarray:
    chunks = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding'):
        chunks.append(embed_batch(texts[i : i + batch_size]))
    return np.vstack(chunks)

In [160]:
docs_for_embeddings = df['text'].tolist()       # raw — เก็บบริบทธรรมชาติ
docs_for_ctfidf = df['joined_tokens'].tolist()  # cleaned — ใช้สกัด keyword

embeddings = embed_corpus(docs_for_embeddings)
print(f"Embeddings shape: {embeddings.shape}")  # expect (N, 768)

Embedding: 100%|██████████| 175/175 [02:19<00:00,  1.25it/s]

Embeddings shape: (2797, 768)


## 3. BERTopic (UMAP + HDBSCAN + c-TF-IDF)

- `random_state=42` ใน UMAP → reproducible
- **`min_cluster_size=15`** (ลดจาก 25) — เพื่อให้แยก topic เล็กๆ ออกจาก bucket ใหญ่
- vectorizer ใช้ regex `[฀-๿]+` จับเฉพาะตัวอักษรไทย
- เก็บ topic `-1` ไว้เป็น outlier/uncertain แทนการบังคับ assign ทุกมาตราเข้า topic

In [161]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=RANDOM_STATE,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,    # ลดจาก 25 → ได้ topic เล็กลงแต่แม่นขึ้น + outlier น้อยลง
    min_samples=3,          # ลดจาก 5 → คลัสเตอร์ที่ขอบเขตนุ่มนวลขึ้น
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    token_pattern=r'[฀-๿]+',
    lowercase=False,
    min_df=2,
    stop_words=custom_stopwords,
)

topic_model = BERTopic(
    language='multilingual',
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs_for_ctfidf, embeddings=embeddings)

df['topic'] = topics

n_outliers = int((df['topic'] == -1).sum())
print(f"Outliers retained: {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")

topic_info = topic_model.get_topic_info()
print(f"Topics found (excl. -1): {(topic_info['Topic'] != -1).sum()}")
topic_info.head(15)

2026-05-04 22:09:39,253 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-04 22:09:49,169 - BERTopic - Dimensionality - Completed ✓
2026-05-04 22:09:49,170 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-04 22:09:49,244 - BERTopic - Cluster - Completed ✓
2026-05-04 22:09:49,251 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-04 22:09:49,326 - BERTopic - Representation - Completed ✓


Outliers retained: 617 (22.1%)
Topics found (excl. -1): 79


,Topic,Count,Name,Representation,Representative_Docs
0,-1,617,-1_ปี_ประชุม_ร่าง_รัฐ,"[ปี, ประชุม, ร่าง, รัฐ, รัฐสภา, ท้องถิ่น, สามั...",[สมาชิก วุฒิสภา คุณสมบัติ ลักษณะ ห้าม ก. คุณสม...
1,0,96,0_ศาลฎีกา_ผู้ดํารงตําแหน่ง_ไต่สวน_ยื่น,"[ศาลฎีกา, ผู้ดํารงตําแหน่ง, ไต่สวน, ยื่น, อัยก...",[ผู้ดํารงตําแหน่ง เมือง จงใจ ยื่น บัญชี รายการ...
2,1,70,1_สิทธิ_เจ้าพนักงาน_ท้องถิ่น_บุคคล,"[สิทธิ, เจ้าพนักงาน, ท้องถิ่น, บุคคล, ทุกข์, เ...",[สิทธิ บุคคล ฟ้อง หน่วย ราชการ นิติบุคคล เนื่อ...
3,2,55,2_ว่าง_แทน_อายุ_สมาชิก,"[ว่าง, แทน, อายุ, สมาชิก, ร้อย, ตำแหน่ง, สภาผู...",[ตําแหน่ง สมาชิก สภาผู้แทนราษฎร ว่าง อายุ สภาผ...
4,3,52,3_วินิจฉัย_ศาลรัฐธรรมนูญ_ศาล_พิจารณา,"[วินิจฉัย, ศาลรัฐธรรมนูญ, ศาล, พิจารณา, คําวิน...",[ศาล คดี ศาล คู่ โต้แย้ง ศาล สมควร 5 วินิจฉัย ...
5,4,47,4_สรรหา_กรรมการ_เลือก_คน,"[สรรหา, กรรมการ, เลือก, คน, เงินแผ่นดิน, ตรวจ,...",[คณะ กรรมการ สรรหา สมาชิก วุฒิ สภาคณะ ประธาน ศ...
6,5,47,5_วินิจฉัย_คณะตุลาการรัฐธรรมนูญ_ร้อง_แย้ง,"[วินิจฉัย, คณะตุลาการรัฐธรรมนูญ, ร้อง, แย้ง, ข...",[สมาชิก วุฒิสภา สมาชิก สภาผู้แทนราษฎร จำนวน สา...
7,6,47,6_สนอง_ประเภท_ธรรมนูญ_พระบรมราชโองการ,"[สนอง, ประเภท, ธรรมนูญ, พระบรมราชโองการ, พิบูล...",[สั่ง นายก รัฐมนตรี สั่งการ อาศัย อำนาจ 17 ธรร...
8,7,44,7_จอมทัพ_เคารพ_ละเมิด_ดำรง,"[จอมทัพ, เคารพ, ละเมิด, ดำรง, ฐานะ, ไทย, กำเนิ...",[องค์ พระมหากษัตริย์ ดำรง ฐานะ เคารพ สักการะ ล...
9,8,44,8_คุก_ตาย_ห้าม_ลา,"[คุก, ตาย, ห้าม, ลา, ลักษณะ, ประมาท, คุณสมบัติ...",[สมาชิกภาพ สมาชิก สภาผู้แทน สิ้นสุด 1 อายุ สภา...


In [162]:
topic_model.visualize_barchart(top_n_topics=10, n_words=8)

In [163]:
topic_model.visualize_topics()

In [164]:
topic_model.visualize_heatmap()

## 4. Validation

ก่อน trust ผลลัพธ์ ตรวจ 2 อย่าง:
1. **Cross-constitution coverage** — topic ที่ดีต้องมีมาตราจาก ≥3 ฉบับ (ไม่ใช่ topic เฉพาะของ รธน. ฉบับเดียว)
2. **เปิดดู Representative_Docs ของ top topics** — ตรวจด้วยตาว่าเรื่องเดียวกันจริง

In [165]:
coverage = (
    df[df['topic'] != -1]
    .groupby('topic')
    .agg(
        n_sections=('section_id', 'count'),
        n_constitutions=('constitution_id', 'nunique'),
        years=('year_th', lambda s: sorted(s.unique().tolist())),
    )
    .reset_index()
    .sort_values('n_sections', ascending=False)
)
coverage['name'] = coverage['topic'].map(topic_info.set_index('Topic')['Name'])
coverage[['topic', 'name', 'n_sections', 'n_constitutions', 'years']].head(15)

,topic,name,n_sections,n_constitutions,years
0,0,0_ศาลฎีกา_ผู้ดํารงตําแหน่ง_ไต่สวน_ยื่น,96,5,"[2517, 2538, 2540, 2550, 2560]"
1,1,1_สิทธิ_เจ้าพนักงาน_ท้องถิ่น_บุคคล,70,12,"[2475, 2489, 2490, 2492, 2511, 2517, 2521, 253..."
2,2,2_ว่าง_แทน_อายุ_สมาชิก,55,18,"[2475, 2489, 2490, 2492, 2495, 2502, 2511, 251..."
3,3,3_วินิจฉัย_ศาลรัฐธรรมนูญ_ศาล_พิจารณา,52,18,"[2489, 2492, 2495, 2502, 2511, 2515, 2517, 251..."
4,4,4_สรรหา_กรรมการ_เลือก_คน,47,13,"[2489, 2511, 2517, 2520, 2534, 2538, 2540, 254..."
5,5,5_วินิจฉัย_คณะตุลาการรัฐธรรมนูญ_ร้อง_แย้ง,47,13,"[2492, 2511, 2517, 2520, 2521, 2534, 2538, 254..."
6,6,6_สนอง_ประเภท_ธรรมนูญ_พระบรมราชโองการ,47,21,"[2475, 2482, 2483, 2485, 2490, 2491, 2492, 249..."
7,7,7_จอมทัพ_เคารพ_ละเมิด_ดำรง,44,16,"[2475, 2489, 2490, 2492, 2502, 2511, 2515, 251..."
8,8,8_คุก_ตาย_ห้าม_ลา,44,13,"[2475, 2489, 2490, 2492, 2495, 2511, 2517, 252..."
9,9,9_พระองค์_สำเร็จ_องคมนตรี_แทน,42,11,"[2475, 2489, 2490, 2492, 2511, 2517, 2521, 253..."


In [166]:
# เปิดดูมาตราตัวอย่างของ top-5 topics
for topic_id in coverage['topic'].head(5):
    name = topic_info.set_index('Topic').loc[topic_id, 'Name']
    print(f"\n=== Topic {topic_id}: {name} ===")
    sample = df[df['topic'] == topic_id].sample(min(10, (df['topic'] == topic_id).sum()), random_state=RANDOM_STATE)
    for _, row in sample.iterrows():
        snippet = row['text'][:250].replace('\n', ' ')
        print(f"  [{row['year_th']}] {snippet}...")


=== Topic 0: 0_ศาลฎีกา_ผู้ดํารงตําแหน่ง_ไต่สวน_ยื่น ===
  [2560] ในการปฏิบัติหน้าที่ ให้องค์กรอิสระร่วมมือและช่วยเหลือกันเพื่อให้บรรลุเป้าหมาย ในการปฏิบัติหน้าที่ของแต่ละองค์กร และถ้าองค์กรอิสระใดเห็นว่ามีผู้กระทําการอันไม่ชอบด้วยกฎหมาย แต่อยู่ในหน้าที่และอํานาจขององค์กรอิสระอื่น ให้แจ้งองค์กรอิสระนั้นทราบเพื่อดํา...
  [2560] ให้มีแผนกคดีอาญาของผู้ดํารงตําแหน่งทางการเมืองในศาลฎีกา โดยองค์คณะ ผู้พิพากษาประกอบด้วยผู้พิพากษาในศาลฎีกาซึ่งดํารงตําแหน่งไม่ต่ํากว่าผู้พิพากษาศาลฎีกาหรือผู้พิพากษา อาวุโสซึ่งเคยดํารงตําแหน่งไม่ต่ํากว่าผู้พิพากษาศาลฎีกา ซึ่งได้รับคัดเลือกโดยที่ประชุ...
  [2550] เพื่อประโยชน์ในการดําเนินการตามหมวดนี้ ให้ผู้ตรวจการแผ่นดินมีอํานาจ หน้าที่เสนอแนะหรือให้คําแนะนําในการจัดทําหรือปรับปรุงประมวลจริยธรรมตามมาตรา 279 วรรคหนึ่ง และส่งเสริมให้ผู้ดํารงตําแหน่งทางการเมือง ข้าราชการ และเจ้าหน้าที่ของรัฐ มีจิตสํานึก ในด้านจ...
  [2560] เพื่อประโยชน์ในการระงับหรือยับยั้งความเสียหายที่อาจเกิดขึ้นแก่การเงินการคลัง ของรัฐ ให้ผู้ว่าการตรวจเงินแผ่นดินเสนอผลการตรวจสอบการกระทําที่ไม่เป็

## 5. Topics Over Time

บอกว่า topic ไหนปรากฏบ่อยแค่ไหนในแต่ละช่วงเวลา (ปี พ.ศ. ของ รธน. แต่ละฉบับ)

**การปรับ:** ใช้ `pd.to_datetime` แปลงปีเป็น datetime เพื่อให้ BERTopic วาง bin ตามจริง (ก่อนหน้านี้ส่งเป็น int → bin จะมี decimal เช่น 2474.911)
และไม่ส่ง `nr_bins` เพื่อให้ใช้ปีจริงเป็น bin (รธน. แต่ละฉบับ = 1 timestamp)

In [167]:
# แปลงปี พ.ศ. → ปี ค.ศ. → datetime เพื่อให้ BERTopic จัดการปีจริงแทน synthetic bins
# ตัด topic -1 ออกจาก time trend เพราะเป็น bucket ของข้อความที่ model ยังไม่มั่นใจ
valid_topic_mask = df['topic'] != -1
year_ce = df.loc[valid_topic_mask, 'year_th'].astype(int) - 543
timestamps = pd.to_datetime(year_ce.astype(str), format='%Y').tolist()

topics_over_time = topic_model.topics_over_time(
    docs=df.loc[valid_topic_mask, 'joined_tokens'].tolist(),
    timestamps=timestamps,
    topics=df.loc[valid_topic_mask, 'topic'].tolist(),
    evolution_tuning=True,
    global_tuning=True,
)
topics_over_time['year_ce'] = pd.to_datetime(topics_over_time['Timestamp']).dt.year
topics_over_time['year_th'] = topics_over_time['year_ce'] + 543
print(f"Constitution years: {topics_over_time['year_th'].nunique()}")
print(f"Year range: {topics_over_time['year_th'].min()} → {topics_over_time['year_th'].max()}")
topics_over_time.head()

31it [00:00, 76.33it/s]

Constitution years: 31
Year range: 2475 → 2564


,Topic,Words,Frequency,Timestamp,year_ce,year_th
0,1,"เคารพ, ช่วยเหลือ, เงื่อนไข, ภาษี, ป้องกัน",1,1932-01-01,1932,2475
1,2,"เต็ม, สภาชิก, ว่าง, ตำแหน่ง, แทน",2,1932-01-01,1932,2475
2,6,"สยาม, ประเภท, ธรรมนูญ, จบ, ราษฎร",2,1932-01-01,1932,2475
3,7,"สยาม, จอมทัพ, แบ่งแยก, กำเนิด, ดำรง",3,1932-01-01,1932,2475
4,8,"เสื่อมเสีย, ประพฤติ, จอมทัพ, เคารพ, ละเมิด",1,1932-01-01,1932,2475


In [168]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)

## 5.5 Interpret Topic Labels

ค่า default `Name` ของ BERTopic เอา top-4 words มาต่อกันด้วย `_` ซึ่งอ่านลำบากเวลาโชว์ผล
ส่วนนี้จึงเพิ่ม label ภาษาไทยแบบ heuristic จาก keyword rules เพื่อช่วยอ่านผลเร็วขึ้น

**ข้อควรระวัง:** label เหล่านี้เป็น interpretive aid ไม่ใช่ ground truth ต้องตรวจ representative documents ก่อนใช้เป็นข้อสรุปเชิงเนื้อหา

In [172]:
# ตั้งชื่อ topic ที่อ่านง่าย — แบบ pattern-matching แทน hardcode topic_id
#
# เหตุผล: topic_id ของ BERTopic เปลี่ยนทุกครั้งที่เรา re-run (เพราะ HDBSCAN labels
# ตามขนาด cluster ที่ขึ้นกับ random seed + parameter) ดังนั้น hardcode {0: 'X', 1: 'Y'}
# จะ map ผิดทันทีถ้าปรับ parameter
#
# วิธีนี้: กำหนด rules แบบ "ถ้า top-N words มี keyword เหล่านี้ → ตั้งชื่อนี้"
# แล้ววน apply กับทุก topic ที่มีอยู่จริง

# (label, required_keywords) — required_keywords ทั้งหมดต้องอยู่ใน representation ของ topic
LABEL_RULES = [
    ('ถวายสัตย์ปฏิญาณ',              ['ปฏิญาณ']),
    ('อภิปรายไม่ไว้วางใจ',            ['อภิปราย', 'ไว้วางใจ']),
    ('ภาวะฉุกเฉิน/อัยการศึก',         ['อัยการศึก']),
    ('ศาลรัฐธรรมนูญ',                ['ศาลรัฐธรรมนูญ']),
    ('คดีอาญา/สิทธิผู้ต้องหา',         ['อาญา', 'คุมขัง']),
    ('สอบสวน/ปล่อยตัว',              ['สอบสวน', 'ปล่อย']),
    ('การสืบราชสมบัติ',              ['สืบ', 'พระปรมาภิไธย']),
    ('การสืบราชสมบัติ',              ['ราชสันตติวงศ์']),
    ('องคมนตรี',                    ['องคมนตรี', 'พระราชอัธยาศัย']),
    ('ผู้สำเร็จราชการแทน',            ['สำเร็จ', 'องคมนตรี']),
    ('พระราชอำนาจ/พระราชกฤษฎีกา',     ['พระราชกฤษฎีกา']),
    ('พระราชสถานะ/จอมทัพ',           ['จอมทัพ']),
    ('อำนาจอธิปไตย/ประมุข',          ['อธิปไตย', 'ประมุข']),
    ('ระบอบประชาธิปไตย',             ['ระบอบ', 'ประชาธิปไตย']),
    ('ครม./นายกรัฐมนตรี',            ['นายก', 'รัฐมนตรี', 'รับสนอง']),
    ('ร่างพระราชบัญญัติ/งบประมาณ',     ['ร่าง', 'พระราชบัญญัติ']),
    ('งบประมาณ/การคลัง',             ['งบ', 'จ่าย', 'คลัง']),
    ('สรรหากรรมการตรวจเงินแผ่นดิน',     ['สรรหา', 'กรรมการ']),
    ('ตรวจเงินแผ่นดิน',               ['เงินแผ่นดิน', 'ตรวจ']),
    ('การไต่สวนผู้ดำรงตำแหน่ง',        ['ไต่สวน']),
    ('ศาลฎีกา/ผู้ดำรงตำแหน่งทางการเมือง', ['ศาลฎีกา', 'ผู้ดํารงตําแหน่ง']),
    ('ศาลยุติธรรม/ศาลปกครอง',         ['ตุลาการ', 'ศาลยุติธรรม']),
    ('ศาลยุติธรรม/ศาลปกครอง',         ['ศาลปกครอง']),
    ('ลงคะแนน/ประชามติ',             ['ประชามติ']),
    ('ลงคะแนน/วาระ',                 ['คะแนน', 'วาระ']),
    ('ประชุมลับ',                    ['ประชุม', 'ลับ']),
    ('เอกสิทธิ์การประชุมสภา',          ['เอกสิทธิ์']),
    ('ประธาน/รองประธานสภา',          ['ประธาน', 'รอง']),
    ('การลาออก/พ้นตำแหน่ง',           ['ลาออก']),
    ('คุณสมบัติต้องห้าม/พ้นตำแหน่ง',     ['ห้าม', 'คุณสมบัติ']),
    ('คุณสมบัติต้องห้าม/พ้นตำแหน่ง',     ['ห้าม', 'ลักษณะ']),
    ('สัญชาติ/คุณสมบัติเลือกตั้ง',       ['สัญชาติ']),
    ('เลือกตั้ง/พรรคการเมือง',          ['เลือกตั้ง', 'พรรค']),
    ('เลือกตั้งเขต/ราษฎร',             ['เขต', 'เลือกตั้ง']),
    ('ตำแหน่งว่าง/แต่งตั้งแทน',         ['ว่าง', 'แทน']),
    ('คณะปฏิวัติ/รัฐประหาร',           ['คณะมนตรีความมั่นคงแห่งชาติ']),
    ('สภานิติบัญญัติแห่งชาติ',          ['สภานิติบัญญัติแห่งชาติ']),
    ('ปฏิรูป/รัฐธรรมนูญใหม่',          ['ปฏิรูป', 'กรรมาธิการ']),
    ('สิทธิเสรีภาพ/ศาสนา',             ['เสรีภาพ', 'ศาสนา']),
    ('สิทธิทางกฎหมาย/วิธีพิจารณา',      ['สิทธิ', 'เจ้าพนักงาน']),
    ('หน้าที่บุคคล/ส่งเสริมรัฐ',         ['ส่งเสริม', 'พัฒนา']),
    ('แรงงาน/เกษตรกร',                ['แรงงาน', 'เกษตรกร']),
    ('หน่วยราชการ/รัฐวิสาหกิจ',         ['รัฐวิสาหกิจ']),
    ('การร้องเรียน/ทุกข์',              ['เรื่องราว', 'ทุกข์']),
    ('ความสัมพันธ์ระหว่างประเทศ',       ['สัมพันธไมตรี']),
    ('ความสัมพันธ์ระหว่างประเทศ',       ['นานา']),
    ('การตีความ/รัฐสภาสองสภา',        ['ตีความ', 'รัฐสภา']),
]


def assign_label(representation: list[str]) -> str | None:
    """Return label ของกฎข้อแรกที่ keyword ครบทุกตัวใน top words ของ topic"""
    rep_set = set(representation)
    for label, required in LABEL_RULES:
        if all(kw in rep_set for kw in required):
            return label
    return None


def add_label_metadata(topic_info_df: pd.DataFrame, manual_labels: dict[int, str]) -> pd.DataFrame:
    """Add review metadata so heuristic labels are not mistaken for validated topics."""
    out = topic_info_df.copy()
    if 'CustomName' not in out.columns:
        out['CustomName'] = out['Name']

    outlier_mask = out['Topic'] == -1
    out.loc[outlier_mask, 'CustomName'] = 'ไม่จัดกลุ่ม/ไม่แน่ชัด'

    out['label_source'] = 'bertopic_default'
    out.loc[out['Topic'].isin(manual_labels), 'label_source'] = 'heuristic_keyword_rule'
    out.loc[outlier_mask, 'label_source'] = 'outlier'

    duplicate_mask = out['CustomName'].duplicated(keep=False) & ~outlier_mask
    out['label_is_duplicate'] = duplicate_mask
    out['label_review_status'] = 'review_required'
    out.loc[out['label_source'] == 'bertopic_default', 'label_review_status'] = 'unlabeled_default_review_required'
    out.loc[outlier_mask, 'label_review_status'] = 'uncertain_unclustered'

    out['topic_display_name'] = out['CustomName']
    out.loc[duplicate_mask, 'topic_display_name'] = (
        out.loc[duplicate_mask, 'CustomName'] + ' [T' + out.loc[duplicate_mask, 'Topic'].astype(str) + ']'
    )
    return out


# Apply rules
manual_labels = {}
for _, row in topic_info.iterrows():
    tid = row['Topic']
    if tid == -1:
        continue
    rep = row['Representation']
    # Representation บางครั้งเป็น string เพราะอ่านจาก CSV; ที่นี่อยู่ใน-memory เป็น list อยู่แล้ว
    if isinstance(rep, str):
        import ast
        rep = ast.literal_eval(rep)
    label = assign_label(rep)
    if label:
        manual_labels[tid] = label

topic_model.set_topic_labels(manual_labels)

print(f"Set custom labels for {len(manual_labels)} / {(topic_info['Topic']!=-1).sum()} topics")
topic_info_named = add_label_metadata(topic_model.get_topic_info(), manual_labels)
topic_info_named[['Topic', 'Count', 'Name', 'CustomName', 'topic_display_name', 'label_source', 'label_review_status', 'label_is_duplicate']].head(25)

Set custom labels for 50 / 79 topics


,Topic,Count,Name,CustomName,topic_display_name,label_source,label_review_status,label_is_duplicate
0,-1,617,-1_ปี_ประชุม_ร่าง_รัฐ,ไม่จัดกลุ่ม/ไม่แน่ชัด,ไม่จัดกลุ่ม/ไม่แน่ชัด,outlier,uncertain_unclustered,False
1,0,96,0_ศาลฎีกา_ผู้ดํารงตําแหน่ง_ไต่สวน_ยื่น,การไต่สวนผู้ดำรงตำแหน่ง,การไต่สวนผู้ดำรงตำแหน่ง,heuristic_keyword_rule,review_required,False
2,1,70,1_สิทธิ_เจ้าพนักงาน_ท้องถิ่น_บุคคล,สิทธิทางกฎหมาย/วิธีพิจารณา,สิทธิทางกฎหมาย/วิธีพิจารณา,heuristic_keyword_rule,review_required,False
3,2,55,2_ว่าง_แทน_อายุ_สมาชิก,ตำแหน่งว่าง/แต่งตั้งแทน,ตำแหน่งว่าง/แต่งตั้งแทน,heuristic_keyword_rule,review_required,False
4,3,52,3_วินิจฉัย_ศาลรัฐธรรมนูญ_ศาล_พิจารณา,ศาลรัฐธรรมนูญ,ศาลรัฐธรรมนูญ [T3],heuristic_keyword_rule,review_required,True
5,4,47,4_สรรหา_กรรมการ_เลือก_คน,สรรหากรรมการตรวจเงินแผ่นดิน,สรรหากรรมการตรวจเงินแผ่นดิน,heuristic_keyword_rule,review_required,False
6,5,47,5_วินิจฉัย_คณะตุลาการรัฐธรรมนูญ_ร้อง_แย้ง,ศาลรัฐธรรมนูญ,ศาลรัฐธรรมนูญ [T5],heuristic_keyword_rule,review_required,True
7,6,47,6_สนอง_ประเภท_ธรรมนูญ_พระบรมราชโองการ,6_สนอง_ประเภท_ธรรมนูญ_พระบรมราชโองการ,6_สนอง_ประเภท_ธรรมนูญ_พระบรมราชโองการ,bertopic_default,unlabeled_default_review_required,False
8,7,44,7_จอมทัพ_เคารพ_ละเมิด_ดำรง,พระราชสถานะ/จอมทัพ,พระราชสถานะ/จอมทัพ,heuristic_keyword_rule,review_required,False
9,8,44,8_คุก_ตาย_ห้าม_ลา,คุณสมบัติต้องห้าม/พ้นตำแหน่ง,คุณสมบัติต้องห้าม/พ้นตำแหน่ง,heuristic_keyword_rule,review_required,False


In [173]:
# Re-render topics_over_time ด้วย custom_labels เพื่อให้ legend อ่านง่าย
topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=10,
    custom_labels=True,
)

## 6. บันทึกผลลัพธ์

In [171]:
# ใช้ topic_info ตัวล่าสุด (หลังตั้ง custom labels) พร้อม metadata เพื่อไม่ให้ heuristic labels ดูเหมือน ground truth
topic_info_final = add_label_metadata(topic_model.get_topic_info(), manual_labels)
topic_name_map = topic_info_final.set_index('Topic')['topic_display_name'].to_dict()
topic_label_source_map = topic_info_final.set_index('Topic')['label_source'].to_dict()
topic_label_status_map = topic_info_final.set_index('Topic')['label_review_status'].to_dict()

out_assignments = df[['section_id', 'constitution_id', 'year_th', 'chapter_number', 'section_number', 'text', 'topic']].copy()
out_assignments['topic_name'] = out_assignments['topic'].map(topic_name_map)
out_assignments['topic_label_source'] = out_assignments['topic'].map(topic_label_source_map)
out_assignments['topic_label_review_status'] = out_assignments['topic'].map(topic_label_status_map)
out_assignments.to_csv('topic_assignments_wangchanberta.csv', index=False, encoding='utf-8-sig')

topic_info_final.to_csv('topic_info_wangchanberta.csv', index=False, encoding='utf-8-sig')
topics_over_time.to_csv('topics_over_time_wangchanberta.csv', index=False, encoding='utf-8-sig')

print('Saved:')
print('  - topic_assignments_wangchanberta.csv')
print('  - topic_info_wangchanberta.csv')
print('  - topics_over_time_wangchanberta.csv')

Saved:
  - topic_assignments_wangchanberta.csv
  - topic_info_wangchanberta.csv
  - topics_over_time_wangchanberta.csv
